## Cell 1 - Imports and paths

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
import numpy as np

from biochemical_stain_assessment.czi_io import read_czi
from biochemical_stain_assessment.registration import (
    enhance_dapi,
    estimate_stain_matrix,
    normalise_abpas,
    register_keypoint,
    save_affine_params,
    save_stain_matrix,
)

DATA_DIR = Path("../data_1_5/JH-311")
OUT_DIR = Path("../registered_output/keypoint_demo")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ABPAS_CZI = DATA_DIR / "AbPAS" / "ITG_Rusha_JH-311_Organoid_AbPAS_HMGU1-1.czi"
DAPI_CZI = DATA_DIR / "DAPI" / "ITG_Rusha_JH-311_Organoid_Hoechst33342_HMGU1-1.czi"

## Cell 2 - Load images and show raw

In [ ]:
abpas, px_abpas, _, cx_a, cy_a, _ = read_czi(ABPAS_CZI)
dapi, px_dapi, _, cx_d, cy_d, _ = read_czi(DAPI_CZI)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(abpas[::4, ::4])
axes[0].set_title("AB-PAS raw (1/4 resolution)")
axes[0].axis("off")
axes[1].imshow(dapi[::4, ::4], cmap="gray")
axes[1].set_title("DAPI raw (1/4 resolution)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## Cell 3 - Resample and coarse crop

In [ ]:
from biochemical_stain_assessment.register_overlay import (
    crop_to_abpas_fov,
    resample_to_pixel_size,
)

dapi_rs = resample_to_pixel_size(dapi, px_dapi, px_abpas)
dapi_crop, r0, c0 = crop_to_abpas_fov(
    dapi_rs,
    abpas.shape[0],
    abpas.shape[1],
    cx_a,
    cy_a,
    cx_d,
    cy_d,
    px_abpas,
)
print(f"After coarse crop: abpas={abpas.shape}, dapi_crop={dapi_crop.shape}")
print(f"DAPI crop offset in resampled image: row={r0}, col={c0}")

## Cell 4 - AB-PAS Macenko normalisation

In [ ]:
stain_mtx = estimate_stain_matrix(abpas)
print("Stain matrix:\n", stain_mtx)
save_stain_matrix(
    stain_mtx,
    OUT_DIR / "stain_matrix_HMGU1-1.json",
    sample_id="HMGU1-1",
    estimated_from=str(ABPAS_CZI),
)

abpas_norm = normalise_abpas(abpas, stain_matrix_target=stain_mtx)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(abpas[::4, ::4])
axes[0].set_title("AB-PAS raw")
axes[0].axis("off")
axes[1].imshow(abpas_norm[::4, ::4])
axes[1].set_title("AB-PAS Macenko-normalised")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## Cell 5 - DAPI contrast enhancement

In [ ]:
dapi_enhanced = enhance_dapi(dapi_crop, clahe_clip_limit=0.01, tophat=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(dapi_crop[::4, ::4], cmap="gray")
axes[0].set_title("DAPI raw (cropped)")
axes[0].axis("off")
axes[1].imshow(dapi_enhanced[::4, ::4], cmap="gray")
axes[1].set_title("DAPI CLAHE-enhanced")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## Cell 6 - Keypoint detection and matching

In [ ]:
import cv2

from biochemical_stain_assessment.registration.keypoint_register import detect_and_match

ref_gray = 1.0 - (
    0.2126 * abpas_norm[:, :, 0]
    + 0.7152 * abpas_norm[:, :, 1]
    + 0.0722 * abpas_norm[:, :, 2]
) / 255.0
ref_gray_ds = ref_gray[::4, ::4].astype(np.float32)
dapi_enhanced_ds = dapi_enhanced[::4, ::4].astype(np.float32)
ref_u8 = (ref_gray_ds * 255).astype(np.uint8)
mov_u8 = (dapi_enhanced_ds * 255).astype(np.uint8)

src_pts, dst_pts = detect_and_match(
    ref_gray_ds,
    dapi_enhanced_ds,
    detector="ORB",
    n_keypoints=500,
)
print(f"Matched point pairs: {len(src_pts)}")

kp1 = [cv2.KeyPoint(float(p[1]), float(p[0]), 1) for p in src_pts[:50]]
kp2 = [cv2.KeyPoint(float(p[1]), float(p[0]), 1) for p in dst_pts[:50]]
matches = [cv2.DMatch(i, i, 0) for i in range(len(kp1))]
vis = cv2.drawMatches(
    ref_u8,
    kp1,
    mov_u8,
    kp2,
    matches,
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)
plt.figure(figsize=(18, 6))
plt.imshow(vis)
plt.title(f"ORB matches (n={len(kp1)} shown)")
plt.axis("off")
plt.tight_layout()
plt.show()

## Cell 7 - Affine fitting and residual plot

In [ ]:
from biochemical_stain_assessment.registration import AffineParams
from biochemical_stain_assessment.registration.keypoint_register import fit_scale_translation

params_ds = fit_scale_translation(src_pts, dst_pts, ransac_threshold=3.0)
residuals = np.array([], dtype=np.float64)
if params_ds is None:
    params_ds = AffineParams(1.0, 0.0, 0.0, 0, "stage_coords", 0.0)
    print("No robust keypoint model found at 1/4 resolution; using stage-coordinate fallback.")
else:
    scale, ty, tx = params_ds.scale, params_ds.ty, params_ds.tx
    pred_row = scale * dst_pts[:, 0] + ty
    pred_col = scale * dst_pts[:, 1] + tx
    residuals = np.sqrt((src_pts[:, 0] - pred_row) ** 2 + (src_pts[:, 1] - pred_col) ** 2)

print(f"AffineParams at 1/4 resolution: {params_ds}")

plt.figure(figsize=(7, 4))
if residuals.size:
    plt.hist(residuals, bins=40)
    plt.axvline(3.0, color="r", linestyle="--", label="RANSAC threshold")
    plt.legend()
else:
    plt.text(0.5, 0.5, "No robust keypoint model", ha="center", va="center")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
plt.xlabel("Residual (px)")
plt.ylabel("Count")
plt.title(f"Match residuals - RMSE={params_ds.rmse:.2f} px, inliers={params_ds.n_inliers}")
plt.tight_layout()
plt.show()

## Cell 8 - Warp DAPI and overlay

In [ ]:
from biochemical_stain_assessment.registration import mutual_crop
from biochemical_stain_assessment.registration.affine_utils import apply_scale_translation

params = AffineParams(
    scale=params_ds.scale,
    tx=params_ds.tx * 4.0,
    ty=params_ds.ty * 4.0,
    n_inliers=params_ds.n_inliers,
    method=params_ds.method,
    rmse=params_ds.rmse * 4.0,
)
dapi_warped = apply_scale_translation(
    dapi_enhanced,
    params.scale,
    params.tx,
    params.ty,
    output_shape=abpas_norm.shape[:2],
)
abpas_final, dapi_final = mutual_crop(abpas_norm, dapi_warped, margin=64)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(abpas_final[::4, ::4])
axes[0].set_title("AB-PAS (normalised)")
axes[0].axis("off")
axes[1].imshow(dapi_final[::4, ::4], cmap="gray")
axes[1].set_title("DAPI (warped)")
axes[1].axis("off")
overlay = abpas_final[::4, ::4].astype(float) / 255.0
overlay[:, :, 2] = np.clip(overlay[:, :, 2] + 0.4 * dapi_final[::4, ::4], 0, 1)
axes[2].imshow(overlay)
axes[2].set_title("Overlay (DAPI in blue)")
axes[2].axis("off")
plt.tight_layout()
plt.show()

## Cell 9 - Save OME-TIFF and params

In [ ]:
from biochemical_stain_assessment.register_overlay import save_ome_tiff, save_preview_png

tiff_path = OUT_DIR / "HMGU1-1_registered.ome.tif"
png_path = OUT_DIR / "HMGU1-1_preview.png"
params_path = OUT_DIR / "HMGU1-1_affine_params.json"

dapi_final_u16 = (dapi_final * 65535).astype(np.uint16)
save_ome_tiff(str(tiff_path), abpas_final, dapi_final_u16, px_abpas, "HMGU1-1")
save_preview_png(str(png_path), abpas_final, dapi_final_u16)
save_affine_params(params, params_path, "HMGU1-1", str(ABPAS_CZI), str(DAPI_CZI))

print("Saved:")
for path in [tiff_path, png_path, params_path]:
    print(f"  {path}")